# XBRL Financial Data Exploration

Exploring the structured financial data embedded in 10-K filings via inline XBRL.
Goal: understand what's available in `ix:nonFraction` tags and how consistent it is
across Apple, Microsoft, and Google — then decide how to integrate it into the RAG pipeline.

In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path
from collections import Counter
import re

def load_filing(company):
    path = Path(f"../data/raw/{company}/10k_latest.html")
    with open(path, "rb") as f:
        return BeautifulSoup(f, "html.parser")

companies = ["apple", "microsoft", "google"]
soups = {c: load_filing(c) for c in companies}
print("Loaded all three filings.")

---
## 1. What's in the ix:nonFraction tags?

Each `ix:nonFraction` tag wraps a single financial number and carries metadata:
- `name` — the XBRL concept (e.g. `us-gaap:Revenue...`)
- `contextRef` — links to a context element that defines the period and entity
- `unitRef` — the unit (USD, shares, etc.)
- `decimals` — precision
- `scale` — multiplier (e.g. scale="6" means value is in millions)

Let's see how many tags each company has and inspect a few.

In [ ]:
# Count ix:nonFraction tags per company
for company in companies:
    tags = soups[company].find_all("ix:nonfraction")
    print(f"{company.upper()}: {len(tags)} ix:nonFraction tags")

In [ ]:
# Inspect the first few tags from Apple to understand the structure
apple_tags = soups["apple"].find_all("ix:nonfraction")[:5]

for tag in apple_tags:
    print(f"name:       {tag.get('name')}")
    print(f"contextRef: {tag.get('contextref')}")
    print(f"unitRef:    {tag.get('unitref')}")
    print(f"decimals:   {tag.get('decimals')}")
    print(f"scale:      {tag.get('scale')}")
    print(f"format:     {tag.get('format')}")
    print(f"value:      {tag.get_text(strip=True)}")
    print()

---
## 2. What concepts are tagged?

XBRL concepts fall into two categories:
- **`us-gaap:`** — standardized US GAAP taxonomy (shared across all companies)
- **Company-specific** (e.g. `aapl:`, `msft:`, `goog:`) — custom extensions

The `us-gaap` concepts are the most useful for cross-company comparison.

In [ ]:
# Break down concepts by namespace for each company
for company in companies:
    tags = soups[company].find_all("ix:nonfraction")
    namespaces = Counter()
    for tag in tags:
        name = tag.get("name", "")
        ns = name.split(":")[0] if ":" in name else "unknown"
        namespaces[ns] += 1
    
    print(f"\n{company.upper()}:")
    for ns, count in namespaces.most_common():
        print(f"  {ns}: {count}")

In [ ]:
# Top 20 most common us-gaap concepts across all companies
# These are the standardized concepts that work for cross-company queries
all_gaap_concepts = Counter()

for company in companies:
    tags = soups[company].find_all("ix:nonfraction")
    for tag in tags:
        name = tag.get("name", "")
        if name.startswith("us-gaap:"):
            all_gaap_concepts[name] += 1

print(f"Total unique us-gaap concepts: {len(all_gaap_concepts)}\n")
print("Top 20 most common (across all 3 companies):")
for concept, count in all_gaap_concepts.most_common(20):
    print(f"  {count}x  {concept}")

In [ ]:
# Which us-gaap concepts appear in ALL three companies?
# These are the best candidates for cross-company RAG queries
concepts_by_company = {}
for company in companies:
    tags = soups[company].find_all("ix:nonfraction")
    concepts_by_company[company] = set(
        tag.get("name") for tag in tags
        if tag.get("name", "").startswith("us-gaap:")
    )

shared = concepts_by_company["apple"] & concepts_by_company["microsoft"] & concepts_by_company["google"]
print(f"us-gaap concepts shared by all 3 companies: {len(shared)}\n")
for concept in sorted(shared):
    print(f"  {concept}")

---
## 3. Extracting actual values

Let's pull out some key financial metrics with their values, periods, and units
to see what a parsed result would look like.

In [ ]:
def extract_xbrl_values(soup):
    """Extract all ix:nonFraction values with their metadata."""
    results = []
    
    # Build a lookup of context elements (they define the time period)
    contexts = {}
    for ctx in soup.find_all("xbrli:context"):
        ctx_id = ctx.get("id")
        period = ctx.find("xbrli:period")
        if period:
            instant = period.find("xbrli:instant")
            start = period.find("xbrli:startdate")
            end = period.find("xbrli:enddate")
            if instant:
                contexts[ctx_id] = {"type": "instant", "date": instant.text}
            elif start and end:
                contexts[ctx_id] = {"type": "duration", "start": start.text, "end": end.text}
    
    for tag in soup.find_all("ix:nonfraction"):
        name = tag.get("name", "")
        raw_value = tag.get_text(strip=True)
        scale = int(tag.get("scale", "0"))
        unit_ref = tag.get("unitref", "")
        ctx_ref = tag.get("contextref", "")
        
        # Parse the numeric value
        try:
            clean_val = raw_value.replace(",", "").replace("(", "-").replace(")", "")
            # Handle dash/em-dash as zero
            if clean_val in ("-", "—", ""):
                numeric_value = 0
            else:
                numeric_value = float(clean_val) * (10 ** scale)
        except ValueError:
            numeric_value = None
        
        context = contexts.get(ctx_ref, {})
        
        results.append({
            "concept": name,
            "value": numeric_value,
            "raw_value": raw_value,
            "scale": scale,
            "unit": unit_ref,
            "context": context,
        })
    
    return results

# Extract from all three
xbrl_data = {}
for company in companies:
    xbrl_data[company] = extract_xbrl_values(soups[company])
    print(f"{company.upper()}: extracted {len(xbrl_data[company])} values")

In [ ]:
# Look at some key metrics — revenue, net income, EPS
KEY_CONCEPTS = [
    "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax",
    "us-gaap:Revenues",
    "us-gaap:NetIncomeLoss",
    "us-gaap:EarningsPerShareBasic",
    "us-gaap:EarningsPerShareDiluted",
    "us-gaap:Assets",
    "us-gaap:Liabilities",
    "us-gaap:StockholdersEquity",
    "us-gaap:CashAndCashEquivalentsAtCarryingValue",
    "us-gaap:OperatingIncomeLoss",
]

for company in companies:
    print(f"\n{'='*60}")
    print(f"{company.upper()}")
    print(f"{'='*60}")
    
    for item in xbrl_data[company]:
        if item["concept"] in KEY_CONCEPTS and item["value"] is not None:
            ctx = item["context"]
            if ctx.get("type") == "instant":
                period = ctx["date"]
            elif ctx.get("type") == "duration":
                period = f"{ctx['start']} to {ctx['end']}"
            else:
                period = "?"
            
            # Format large numbers for readability
            val = item["value"]
            if abs(val) >= 1e9:
                val_str = f"${val/1e9:,.1f}B"
            elif abs(val) >= 1e6:
                val_str = f"${val/1e6:,.1f}M"
            elif item["unit"] and "share" in item["unit"].lower():
                val_str = f"${val:,.2f}"
            else:
                val_str = f"${val:,.0f}"
            
            # Readable concept name
            short_name = item["concept"].split(":")[1]
            print(f"  {short_name:<55} {val_str:>12}  ({period})")

---
## 4. RAG integration — what would this look like?

If we convert XBRL values into natural language sentences, they can be chunked
and embedded alongside narrative text — no changes to the retrieval pipeline needed.

In [ ]:
def humanize_concept(concept_name):
    """Convert XBRL concept names to readable labels."""
    short = concept_name.split(":")[1]
    # Insert spaces before capital letters
    readable = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", short)
    return readable

def xbrl_to_sentences(company, values):
    """Convert extracted XBRL values into natural language sentences for RAG."""
    sentences = []
    for item in values:
        if item["value"] is None:
            continue
        # Only include us-gaap concepts for now
        if not item["concept"].startswith("us-gaap:"):
            continue
            
        ctx = item["context"]
        label = humanize_concept(item["concept"])
        val = item["value"]
        
        # Format the value
        if abs(val) >= 1e9:
            val_str = f"${val/1e9:,.1f} billion"
        elif abs(val) >= 1e6:
            val_str = f"${val/1e6:,.1f} million"
        else:
            val_str = f"${val:,.2f}"
        
        # Format the period
        if ctx.get("type") == "instant":
            period_str = f"as of {ctx['date']}"
        elif ctx.get("type") == "duration":
            period_str = f"for the period {ctx['start']} to {ctx['end']}"
        else:
            continue
        
        sentence = f"{company.title()}'s {label} was {val_str} {period_str}."
        sentences.append(sentence)
    
    return sentences

# Preview what the RAG-ready output looks like
for company in companies:
    sentences = xbrl_to_sentences(company, xbrl_data[company])
    print(f"\n{company.upper()} — {len(sentences)} sentences generated")
    print("Sample (first 10):")
    for s in sentences[:10]:
        print(f"  {s}")

---
## 5. Curated approach — filtering noise and resolving ambiguity

The raw extraction has two problems:
1. **Too many values** — segment breakdowns, quarterly subtotals, and obscure line items add noise
2. **Ambiguity** — the same concept appears multiple times per period (total vs segment) with no label

The context elements hold segment info that disambiguates these. Let's build a cleaner extraction.

In [ ]:
# First, let's understand the segment structure in context elements
# Contexts WITHOUT segments = company-level totals
# Contexts WITH segments = breakdowns (by product, geography, etc.)

def build_context_lookup(soup):
    """Build a full context lookup including segment info."""
    contexts = {}
    for ctx in soup.find_all("xbrli:context"):
        ctx_id = ctx.get("id")
        
        # Period
        period = ctx.find("xbrli:period")
        period_info = {}
        if period:
            instant = period.find("xbrli:instant")
            start = period.find("xbrli:startdate")
            end = period.find("xbrli:enddate")
            if instant:
                period_info = {"type": "instant", "date": instant.text}
            elif start and end:
                period_info = {"type": "duration", "start": start.text, "end": end.text}
        
        # Segments (this is what distinguishes total vs breakdown)
        segments = []
        segment_el = ctx.find("xbrli:segment")
        if segment_el:
            for member in segment_el.find_all("xbrldi:explicitmember"):
                dim = member.get("dimension", "")
                val = member.text
                segments.append({"dimension": dim, "value": val})
        
        contexts[ctx_id] = {"period": period_info, "segments": segments}
    
    return contexts

# Show what segments look like for Apple's revenue entries
apple_contexts = build_context_lookup(soups["apple"])
revenue_tags = [t for t in soups["apple"].find_all("ix:nonfraction") if "Revenue" in t.get("name", "")]

print("Apple revenue entries with their context:\n")
for tag in revenue_tags:
    ctx = apple_contexts.get(tag.get("contextref"), {})
    segments = ctx.get("segments", [])
    period = ctx.get("period", {})
    
    if period.get("type") == "duration":
        period_str = f"{period['start']} to {period['end']}"
    else:
        period_str = period.get("date", "?")
    
    seg_str = "TOTAL" if not segments else ", ".join(s["value"].split(":")[-1] for s in segments)
    value = tag.get_text(strip=True)
    scale = int(tag.get("scale", "0"))
    
    print(f"  {seg_str:<30} {value:>12} (scale={scale})  {period_str}")

In [ ]:
# Now build the curated extraction:
# 1. Only keep high-value us-gaap concepts
# 2. Include segment labels when present
# 3. Produce cleaner, disambiguated sentences

CURATED_CONCEPTS = {
    # Income statement
    "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax": "Revenue",
    "us-gaap:Revenues": "Revenue",
    "us-gaap:CostOfGoodsAndServicesSold": "Cost of Revenue",
    "us-gaap:GrossProfit": "Gross Profit",
    "us-gaap:OperatingIncomeLoss": "Operating Income",
    "us-gaap:NetIncomeLoss": "Net Income",
    "us-gaap:EarningsPerShareBasic": "Earnings Per Share (Basic)",
    "us-gaap:EarningsPerShareDiluted": "Earnings Per Share (Diluted)",
    
    # Balance sheet
    "us-gaap:Assets": "Total Assets",
    "us-gaap:Liabilities": "Total Liabilities",
    "us-gaap:StockholdersEquity": "Stockholders Equity",
    "us-gaap:CashAndCashEquivalentsAtCarryingValue": "Cash and Cash Equivalents",
    "us-gaap:LongTermDebt": "Long-Term Debt",
    "us-gaap:LongTermDebtNoncurrent": "Long-Term Debt (Non-Current)",
    "us-gaap:ShortTermInvestments": "Short-Term Investments",
    "us-gaap:AccountsReceivableNetCurrent": "Accounts Receivable",
    "us-gaap:Goodwill": "Goodwill",
    
    # Cash flow
    "us-gaap:NetCashProvidedByUsedInOperatingActivities": "Operating Cash Flow",
    "us-gaap:NetCashProvidedByUsedInInvestingActivities": "Investing Cash Flow",
    "us-gaap:NetCashProvidedByUsedInFinancingActivities": "Financing Cash Flow",
    
    # Shares
    "us-gaap:WeightedAverageNumberOfSharesOutstandingBasic": "Shares Outstanding (Basic)",
    "us-gaap:WeightedAverageNumberOfDilutedSharesOutstanding": "Shares Outstanding (Diluted)",
    "us-gaap:CommonStockSharesOutstanding": "Common Shares Outstanding",
    
    # R&D
    "us-gaap:ResearchAndDevelopmentExpense": "R&D Expense",
}

def extract_curated(soup, company):
    """Extract only high-value financial metrics with segment disambiguation."""
    contexts = build_context_lookup(soup)
    sentences = []
    
    for tag in soup.find_all("ix:nonfraction"):
        concept = tag.get("name", "")
        if concept not in CURATED_CONCEPTS:
            continue
        
        label = CURATED_CONCEPTS[concept]
        raw_value = tag.get_text(strip=True)
        scale = int(tag.get("scale", "0"))
        ctx_ref = tag.get("contextref", "")
        ctx = contexts.get(ctx_ref, {})
        
        # Parse value
        try:
            clean_val = raw_value.replace(",", "").replace("(", "-").replace(")", "")
            if clean_val in ("-", "—", ""):
                continue
            numeric_value = float(clean_val) * (10 ** scale)
        except ValueError:
            continue
        
        # Format value
        if abs(numeric_value) >= 1e9:
            val_str = f"${numeric_value/1e9:,.1f} billion"
        elif abs(numeric_value) >= 1e6:
            val_str = f"${numeric_value/1e6:,.1f} million"
        elif "Share" in label or "Earnings Per" in label:
            val_str = f"${numeric_value:,.2f}"
        else:
            val_str = f"${numeric_value:,.0f}"
        
        # Format period
        period = ctx.get("period", {})
        if period.get("type") == "instant":
            period_str = f"as of {period['date']}"
        elif period.get("type") == "duration":
            period_str = f"for the period {period['start']} to {period['end']}"
        else:
            continue
        
        # Build segment qualifier (this resolves the ambiguity)
        segments = ctx.get("segments", [])
        if segments:
            seg_parts = []
            for s in segments:
                # Clean up the segment value: "aapl:IPhoneMember" -> "iPhone"
                raw_seg = s["value"].split(":")[-1]
                clean_seg = raw_seg.replace("Member", "")
                clean_seg = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", clean_seg)
                seg_parts.append(clean_seg)
            segment_str = f" ({', '.join(seg_parts)})"
        else:
            segment_str = ""
        
        sentence = f"{company.title()}'s {label}{segment_str} was {val_str} {period_str}."
        sentences.append(sentence)
    
    return sentences

# Run curated extraction on all companies
for company in companies:
    sentences = extract_curated(soups[company], company)
    print(f"\n{company.upper()} — {len(sentences)} curated sentences (vs {len(xbrl_to_sentences(company, xbrl_data[company]))} raw)")
    print("\nSample:")
    for s in sentences[:15]:
        print(f"  {s}")

---
## Summary

Questions to answer after running this notebook:

1. **Coverage** — Are the key financial metrics (revenue, net income, EPS, assets) reliably tagged across all three companies?
2. **Consistency** — Do they use the same `us-gaap` concept names, or do some companies use custom extensions?
3. **Noise reduction** — How much did the curated approach cut compared to raw? Is the remaining set clean enough?
4. **Disambiguation** — Do the segment labels (e.g. "iPhone", "Product", "Service") make it clear which number is which?